# 강의 04 · 실습 3 — 서브그래프 모듈화 · (3.5) 디버깅

## 1. 문제상황

- 온라인 서점 고객센터는 배송 지연, 도서 파손, 환불 문의 메일을 받습니다.
- 분류 절차는 다른 팀이 이미 자식 그래프로 만들어 두었는데, 그 그래프의 상태 키(text·label·urgent)가 고객센터 그래프의 키(email·category·priority)와 다릅니다.
- 고객센터 그래프는 분류 결과뿐 아니라 긴급 여부도 받아서 담당자 배정에 써야 합니다.
- 키 이름이 다르다고 분류 절차를 고객센터 그래프에 다시 옮겨 적으면, 절차가 바뀔 때마다 두 곳을 고쳐야 합니다.

## 2. 문제와 목표

- **문제**: 아래 「6. 코드 — 스텝바이스텝」의 코드는 이 목표를 잘못 구현한 것입니다. 문법 오류 없이 실행되지만 동작이 요구사항과 어긋나며, 결함 세 개(증상이 서로 다름)를 모두 찾아 고쳐야 「7. 실행 결과 확인」이 통과됩니다.
- **목표**
  - 다른 팀의 분류 그래프를 고치지 않고 자식 그래프로 컴파일해, 고객센터 부모 그래프의 노드로 래퍼(wrapper) 함수를 등록합니다.
    - 래퍼 함수가 번역하는 키: 넣을 때 `email` → `text`, 되받을 때 `label` → `category`, `urgent`(참/거짓) → `priority`(긴급/일반)
- **목표 달성 여부의 판정 기준**:
  - 찢어진 책이 급하다는 메일과 배송 조회 메일을 넣었을 때,
  - 첫 메일은 「파손 담당자에게 긴급 배정」, 둘째 메일은 「배송 담당자에게 일반 배정」이라는 배정 메시지가 만들어지는 것을 실행 결과에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex03_s3_diagram.svg)

## 4. 단계별 요구사항

1. **상태를 정의합니다.**
    - 부모 상태 `TicketState`는 메일 본문(`email`), 분류 결과(`category`), 긴급도(`priority`), 배정 결과(`handled`) 키 네 개를 가집니다.
    - 자식 전용 상태 `SubState`는 분류할 글(`text`), 분류 결과(`label`), 긴급 여부(`urgent`, 참/거짓) 키 세 개를 가집니다.
2. **자식 노드 함수를 만듭니다.**
    - classify 노드는 자식 상태의 `text`를 읽어 배송·파손·환불 중 한 단어를 `label` 키에 씁니다.
    - judge_urgent 노드는 분류 결과가 파손이거나 본문에 「급」이 들어 있으면 `urgent` 키에 `True`를, 아니면 `False`를 씁니다.
    - 모델을 부르지 않는 규칙 노드입니다.
3. **부모 노드 함수를 만듭니다.**
    - handle 노드는 부모 상태의 분류 결과와 긴급도를 읽어 「<분류> 담당자에게 <긴급도> 배정」 문장을 `handled` 키에 씁니다.
4. **자식 그래프를 구성하고 컴파일합니다.**
    - 자식 상태로 빌더를 열고 classify → judge_urgent를 이어 컴파일합니다.
5. **래퍼 함수를 만들고 부모 그래프에 노드를 등록합니다.**
    - 래퍼 함수는 부모의 `email`을 자식의 `text`로 넣어 컴파일된 자식을 실행하고, 돌아온 `label`을 `category`로, `urgent`가 참이면 「긴급」 아니면 「일반」을 `priority`로 돌려줍니다.
    - 부모 그래프에 래퍼 함수를 classifier 이름으로, handle 노드를 handle 이름으로 등록합니다.
6. **엣지를 연결합니다.**
    - START → classifier → handle → END를 고정 엣지로 연결합니다.
7. **그래프를 컴파일하고 실행합니다.**
    - 찢어진 책 메일과 배송 조회 메일을 차례로 넣고, 노드가 하나 끝날 때마다 바뀐 키를 출력합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 자식 그래프도 같은 다섯 단계로 세우며, 래퍼 함수로 자식을 부모의 노드로 등록하는 일은 ③ 노드 등록 단계의 확장입니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 부모 상태와 자식 전용 상태의 키를 선언합니다 | `class TicketState(TypedDict)`, `class SubState(TypedDict)` | 1 |
| ② 노드 함수 정의 | 상태를 받아 바뀐 키만 돌려주는 함수를 만듭니다 | `def classify(state) -> dict` | 2, 3 |
| ③ 그래프 빌더 생성과 노드 등록 | 자식 그래프를 세워 컴파일하고, 래퍼 함수와 부모 노드를 부모 그래프에 등록합니다 | `StateGraph(SubState)`, 자식 그래프 호출, `add_node("classifier", call_child)` | 4, 5 |
| ④ 엣지 연결 | 부모 노드 사이의 순서를 정합니다 | `add_edge` | 6 |
| ⑤ 컴파일과 실행 | 부모 그래프를 컴파일하고 입력을 넣어 실행합니다 | `compile()`, `stream()` | 7 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from typing import TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

### 단계 ① — 상태 정의 (요구사항 1)

부모 상태와 자식 전용 상태를 선언합니다. 자식 상태의 키 세 개(text·label·urgent) 중 키 두 개(label·urgent)가 부모로 되돌아와야 합니다.

In [ ]:
class TicketState(TypedDict):
    email: str      # 받은 메일 본문
    category: str   # 분류 결과 (배송 / 파손 / 환불)
    priority: str   # 긴급도 (긴급 / 일반)
    handled: str    # 배정 결과


class SubState(TypedDict):   # 다른 팀의 자식 그래프가 쓰는 상태. 키 이름이 부모와 다르다
    text: str      # 분류할 글
    label: str     # 분류 결과
    urgent: bool   # 긴급 여부


print("부모 상태의 키:", list(TicketState.__annotations__))
print("자식 상태의 키:", list(SubState.__annotations__))

### 단계 ② — 노드 함수 정의 (요구사항 2, 3)

- 자식 노드 둘은 자식 상태의 이름(text·label·urgent)으로 읽고 씁니다. judge_urgent는 모델을 부르지 않는 규칙 노드입니다.
- 부모 노드 handle은 부모 상태의 이름(category·priority)으로 읽습니다.

In [ ]:
def classify(state: SubState) -> dict:
    """자식 상태의 text를 읽어 배송·파손·환불 중 하나로 분류한다."""
    res = llm.invoke([
        SystemMessage("온라인 서점 고객 메일을 배송, 파손, 환불 중 하나로 분류한다. 다른 말 없이 단어 하나만 답한다."),
        HumanMessage(state["text"]),
    ])
    return {"label": res.content.strip()}


def judge_urgent(state: SubState) -> dict:
    """분류가 파손이거나 본문에 '급'이 있으면 긴급으로 본다 (규칙)."""
    urgent = state["label"] == "파손" or "급" in state["text"]
    return {"urgent": urgent}


def handle(state: TicketState) -> dict:
    """분류 결과와 긴급도에 맞는 담당자에게 배정한다 (여기서는 배정 메시지 작성으로 대신한다)."""
    return {"handled": f"{state['priority']} 담당자에게 {state['category']} 배정"}

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 4, 5)

③-a에서 자식 그래프를 세워 컴파일하고, ③-b에서 래퍼 함수를 만들어 부모 그래프에 등록합니다.

#### 단계 ③-a — 자식 그래프 구성과 컴파일 (요구사항 4)

In [ ]:
child = StateGraph(SubState)
child.add_node("classify", classify)
child.add_node("judge_urgent", judge_urgent)
child.add_edge(START, "classify")
child.add_edge("classify", "judge_urgent")
child.add_edge("judge_urgent", END)
CHILD = child.compile()

print("자식의 노드:", list(child.nodes))

#### 단계 ③-b — 래퍼 함수 정의와 부모 그래프 노드 등록 (요구사항 5)

래퍼 함수는 넣을 때 한 키(email → text), 되받을 때 두 키(label → category, urgent → priority)를 번역합니다. urgent는 참/거짓이므로 되받을 때 「긴급」·「일반」 문자열로 바꿉니다.

In [ ]:
def call_child(state: TicketState) -> dict:
    """부모 상태를 자식 상태로 번역해 넣고, 결과 두 키의 이름을 부모 키로 되돌린다."""
    out = classify({"text": state["email"]})                # email -> text
    print(f"      [자식] label = {out['label']!r}, urgent = {out.get('urgent')}")
    return {"category": out["label"],                       # label -> category
            "priority": "긴급" if out.get("urgent") else "일반"}   # urgent -> priority


parent = StateGraph(TicketState)
parent.add_node("classifier", call_child)
parent.add_node("handle", handle)

print("부모의 노드:", list(parent.nodes))

### 단계 ④ — 엣지 연결 (요구사항 6)

In [ ]:
parent.add_edge(START, "classifier")
parent.add_edge("classifier", END)
parent.add_edge("handle", END)

print("부모 그래프의 연결을 마쳤습니다.")

### 단계 ⑤ — 컴파일과 실행 (요구사항 7)

In [ ]:
graph = parent.compile()

EMAILS = [
    "주문한 책이 찢어진 채로 왔습니다. 내일 선물해야 해서 급합니다.",
    "배송 조회가 되지 않습니다. 언제쯤 도착하나요?",
]

for i, email in enumerate(EMAILS, 1):
    print(f"=== {i}번 메일: {email[:24]}... ===")
    final = {"email": email}
    for step in graph.stream({"email": email}, stream_mode="updates"):
        for node, patch in step.items():
            print(f"  [{node}] -> {patch}")
            final.update(patch)
    print(f"  [최종 상태] category={final.get('category')!r} priority={final.get('priority')!r} handled={final.get('handled')!r}")
    print()

## 7. 실행 결과 확인

세 결함을 모두 고친 뒤의 실행 결과에서 다음 세 가지를 확인합니다.

1. 두 메일 모두 `[classifier]` 줄 다음에 `[handle]` 줄이 출력되고, 최종 상태의 `handled`가 `None`이 아닙니다.
2. 1번 메일의 `priority`는 「긴급」, 2번 메일의 `priority`는 「일반」입니다. 두 메일의 `priority`가 같다면 자식 그래프가 끝까지 실행되지 않은 것입니다.
3. 배정 메시지는 「<분류> 담당자에게 <긴급도> 배정」 순서입니다. 1번 메일은 「파손 담당자에게 긴급 배정」, 2번 메일은 「배송 담당자에게 일반 배정」입니다.